# 02 - IV Smile

Implied volatility across strikes for one expiry.

Classic shape: a "smile" or "skew" — out-of-the-money options have higher IV
than at-the-money.

Reads `implied_volatility` for the nearest expiry.


## Setup

In [ ]:
import sys, sqlite3
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / "database").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))

OPT_DB = _p / "database" / "options.db"
MKT_DB = _p / "database" / "market.db"

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

def opt(q, **p):
    with sqlite3.connect(OPT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

def mkt(q, **p):
    with sqlite3.connect(MKT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

print("options.db:", OPT_DB)
print("market.db :", MKT_DB)

## 1. Load IV for nearest expiry

In [ ]:
iv = opt("""
    SELECT * FROM implied_volatility
    WHERE underlying='ETH'
      AND ts = (SELECT MAX(ts) FROM implied_volatility WHERE underlying='ETH')
    ORDER BY expiry, strike
""")
print("rows:", len(iv))
iv["dte"] = iv.dte.astype(float)
iv = iv[iv.dte > 0]
valid_expiries = iv.groupby("expiry").size().sort_index()
print("expiries available:")
print(valid_expiries)

## 2. Pick one expiry, plot the smile

In [ ]:
if not iv.empty:
    target = iv.expiry.iloc[0]
    snap = iv[iv.expiry == target]
    dte_days = round(float(snap.dte.iloc[0]), 1)

    fig, ax = plt.subplots()
    for kind, color in [("C", "seagreen"), ("P", "crimson")]:
        s = snap[snap["kind"] == kind].sort_values("strike")
        ax.plot(s.strike, s.iv * 100, "o-", color=color,
                label="Calls" if kind == "C" else "Puts")
    ax.set_title(f"IV smile — expiry {pd.to_datetime(target, unit='ms').date()}  ({dte_days}d)")
    ax.set_xlabel("strike")
    ax.set_ylabel("IV %")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. ATM IV + skew

In [ ]:
if not iv.empty:
    spot = iv.iv.iloc[0]
    # ATM = strike nearest spot - we use underlying_px if present, else skip
    oc = opt("""
        SELECT underlying_px FROM option_chain
        WHERE underlying='ETH'
        ORDER BY ts DESC LIMIT 1
    """)
    spot = float(oc.underlying_px.iloc[0]) if not oc.empty and oc.underlying_px.iloc[0] else None

    if spot:
        target = iv.expiry.iloc[0]
        snap = iv[iv.expiry == target].copy()
        atm_row = snap.iloc[(snap.strike - spot).abs().argsort()[:1]]
        print(f"spot  = {spot:.2f}")
        print(f"ATM strike ~ {atm_row.strike.iloc[0]:.0f}")
        print(f"ATM IV = {atm_row.iv.iloc[0]*100:.2f}%")